In [1]:

"""Audit ACI failure-point local errors using existing online trajectories.

No ACI rerun or synthetic fallback.

Quantile-crossing observations are excluded, not counted as failures.
A local window is eligible only when all 24 intervals are valid.

EWMA at i uses all valid local errors up to i, including LE_i,
with weights proportional to M**(i-k), using original time indices.
"""

from pathlib import Path
import numpy as np
import pandas as pd

ALPHA = 0.10
WINDOW = 24
M_DECAY = 0.95


def audit_one(path: Path, alpha=ALPHA, window=WINDOW, decay=M_DECAY):

    df = pd.read_csv(path)

    required = {'y_true', 'y_l_pred', 'y_u_pred'}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f'{path}: missing columns {sorted(missing)}'
        )

    if not (0 < alpha < 1 and window >= 2 and 0 < decay <= 1):
        raise ValueError(
            'Require 0<alpha<1, window>=2, 0<decay<=1'
        )

    if len(df) < window:
        raise ValueError(
            f'{path}: only {len(df)} rows (< window={window})'
        )

    for col in required:
        df[col] = pd.to_numeric(df[col], errors='raise')

    if not np.isfinite(
        df[list(required)].to_numpy(dtype=float)
    ).all():
        raise ValueError(
            f'{path}: missing or nonfinite interval observations'
        )

    # --------------------------------------------------------
    # FIX 1: Identify quantile crossing.
    # Do NOT count crossed intervals as failures.
    # --------------------------------------------------------

    y = df['y_true'].to_numpy(dtype=float)
    low = df['y_l_pred'].to_numpy(dtype=float)
    high = df['y_u_pred'].to_numpy(dtype=float)

    crossed = low > high
    valid = ~crossed

    crossed_count = int(crossed.sum())

    if crossed_count:
        print(
            f'WARNING {path.name}: '
            f'{crossed_count} crossed intervals excluded.'
        )

    # --------------------------------------------------------
    # Validate chronological order
    # --------------------------------------------------------

    if 'step' in df.columns:
        steps = pd.to_numeric(
            df['step'], errors='raise'
        ).to_numpy()

        if not np.all(np.diff(steps) == 1):
            raise ValueError(
                f'{path}: step must be consecutive and sorted'
            )
    else:
        steps = np.arange(len(df))

    # --------------------------------------------------------
    # FIX 2: Compute coverage errors.
    # Use stored online ACI errors if available.
    # Ignore crossed intervals.
    # --------------------------------------------------------

    computed_error = (
        (y < low) | (y > high)
    ).astype(np.int8)

    error = computed_error.copy()

    if 'errSeq' in df.columns:

        stored = pd.to_numeric(
            df['errSeq'], errors='raise'
        ).to_numpy(dtype=float)

        if (
            not np.isfinite(stored).all()
            or not np.isin(stored, [0, 1]).all()
        ):
            raise ValueError(
                f'{path}: errSeq contains invalid values'
            )

        mismatch = int(
            np.count_nonzero(
                (stored != computed_error) & valid
            )
        )

        if mismatch:
            print(
                f'WARNING {path.name}: '
                f'{mismatch} valid rows disagree with '
                f'saved interval coverage. '
                f'Using recorded errSeq.'
            )

        error = stored.astype(np.int8)

    # Crossed intervals are not evaluated.
    error = error.astype(float)
    error[crossed] = np.nan

    # --------------------------------------------------------
    # FIX 3: Full valid trailing windows only.
    #
    # If any observation in the last 24 is crossed,
    # the entire local window is ineligible.
    # --------------------------------------------------------

    covered = 1.0 - error

    local_coverage = (
        pd.Series(covered)
        .rolling(window, min_periods=window)
        .mean()
        .to_numpy()
    )

    local_error = np.maximum(
        (1 - alpha) - local_coverage,
        0.0
    )

    # Invalid or incomplete windows remain NaN.

    # --------------------------------------------------------
    # FIX 4: Exponentially weighted LE baseline.
    #
    # Original time indices are preserved.
    # Only valid local errors enter the baseline.
    #
    # LEbar_i =
    #   sum(valid k <= i) M**(i-k) * LE_k
    #   ----------------------------------
    #   sum(valid k <= i) M**(i-k)
    #
    # Current LE_i is included if valid.
    # --------------------------------------------------------

    baseline = np.full(len(df), np.nan)

    numerator = 0.0
    denominator = 0.0

    for i in range(window - 1, len(df)):

        numerator *= decay
        denominator *= decay

        if np.isfinite(local_error[i]):

            numerator += local_error[i]
            denominator += 1.0

            baseline[i] = numerator / denominator

    # --------------------------------------------------------
    # FIX 5: Eligible failures only.
    #
    # Exclude crossed observations and invalid windows.
    # --------------------------------------------------------

    eligible = (
        (np.arange(len(df)) >= window - 1)
        & valid
        & np.isfinite(local_error)
        & np.isfinite(baseline)
    )

    failure = (error == 1) & eligible

    # STRICT >
    above = (
        local_error[failure] > baseline[failure]
    )

    gap = (
        local_error[failure] - baseline[failure]
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    result = {
        'Dataset': path.stem.replace('__ACI', ''),
        'N_total': int(len(df)),
        'N_crossed_intervals': crossed_count,
        'N_failures_all': int(
            np.nansum(error == 1)
        ),
        'N_failures_eligible': int(failure.sum()),
        'N_LE_above_LEbar': int(above.sum()),
        'Pct_above': (
            100 * float(above.mean())
            if failure.any() else np.nan
        ),
        'Mean_LE_failure': (
            float(local_error[failure].mean())
            if failure.any() else np.nan
        ),
        'Mean_LEbar_failure': (
            float(baseline[failure].mean())
            if failure.any() else np.nan
        ),
        'Delta': (
            float(gap.mean())
            if failure.any() else np.nan
        ),
    }

    details = pd.DataFrame({
        'step': steps[failure],
        'LE_L': local_error[failure],
        'LEbar_L': baseline[failure],
        'Delta': gap,
        'LE_gt_LEbar': above.astype(int),
    })

    return result, details


def main():

    # Original nine dataset paths unchanged.

    files = [
        Path('../01. Exp_6.3_ACI_vs_QR_Experiment/trajectories/LSTM/50Hertz__ACI.csv'),

        Path('../01. Exp_6.3_ACI_vs_QR_Experiment/trajectories/LSTM/Amprion__ACI.csv'),

        Path('../01. Exp_6.3_ACI_vs_QR_Experiment/trajectories/LSTM/TenneTTSO__ACI.csv'),

        Path('../01. Exp_6.3_ACI_vs_QR_Experiment/trajectories/LSTM/TransnetBW__ACI.csv'),

        Path('../01. Exp_6.3_ACI_vs_QR_Experiment/trajectories/LSTM/hydropower_dataset__ACI.csv'),

        Path('../01. Exp_6.3_ACI_vs_QR_Experiment/trajectories/LSTM/Location1__ACI.csv'),

        Path('../01. Exp_6.3_ACI_vs_QR_Experiment/trajectories/LSTM/Location2__ACI.csv'),

        Path('../01. Exp_6.3_ACI_vs_QR_Experiment/trajectories/LSTM/Plant_1_Generation_Data__ACI.csv'),

        Path('../01. Exp_6.3_ACI_vs_QR_Experiment/trajectories/LSTM/Plant_2_Generation_Data__ACI.csv'),
    ]

    output_dir = Path('./aci_failure_audit')

    alpha = ALPHA
    window = WINDOW
    decay = M_DECAY

    for p in files:
        if not p.is_file():
            raise FileNotFoundError(p)

    output_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    for p in files:

        print(f'\nProcessing: {p.name}')

        summary, details = audit_one(
            p, alpha, window, decay
        )

        rows.append(summary)

        details.to_csv(
            output_dir / f'{p.stem}_failure_points.csv',
            index=False
        )

    res = pd.DataFrame(rows)

    res.to_csv(
        output_dir / 'aci_failure_summary.csv',
        index=False
    )

    print(
        res.to_string(
            index=False,
            float_format=lambda x: f'{x:.6f}'
        )
    )

    if len(res) > 1:

        total_above = int(
            res['N_LE_above_LEbar'].sum()
        )

        total_failures = int(
            res['N_failures_eligible'].sum()
        )

        weighted_pct = (
            100 * total_above / total_failures
            if total_failures else np.nan
        )

        # Pooled mean difference across all failures.
        pooled_delta = (
            np.average(
                res.loc[
                    res.N_failures_eligible > 0,
                    'Delta'
                ],
                weights=res.loc[
                    res.N_failures_eligible > 0,
                    'N_failures_eligible'
                ]
            )
            if total_failures else np.nan
        )

        print(
            f'\nPooled count: {total_above}'
            f'/{total_failures}'
            f' ({weighted_pct:.2f}%).'
        )

        print(
            f'Pooled mean Delta: {pooled_delta:.6f}'
        )

    print(
        f'\nSaved CSV outputs to: '
        f'{output_dir.resolve()}'
    )


if __name__ == '__main__':
    main()



Processing: 50Hertz__ACI.csv
WARNING 50Hertz__ACI.csv: 13 crossed intervals excluded.

Processing: Amprion__ACI.csv
WARNING Amprion__ACI.csv: 12 crossed intervals excluded.

Processing: TenneTTSO__ACI.csv
WARNING TenneTTSO__ACI.csv: 8 crossed intervals excluded.

Processing: TransnetBW__ACI.csv
WARNING TransnetBW__ACI.csv: 1 crossed intervals excluded.

Processing: hydropower_dataset__ACI.csv

Processing: Location1__ACI.csv
WARNING Location1__ACI.csv: 7 crossed intervals excluded.

Processing: Location2__ACI.csv
WARNING Location2__ACI.csv: 9 crossed intervals excluded.

Processing: Plant_1_Generation_Data__ACI.csv
WARNING Plant_1_Generation_Data__ACI.csv: 27 crossed intervals excluded.
WARNING Plant_1_Generation_Data__ACI.csv: 1 valid rows disagree with saved interval coverage. Using recorded errSeq.

Processing: Plant_2_Generation_Data__ACI.csv
WARNING Plant_2_Generation_Data__ACI.csv: 1 crossed intervals excluded.
WARNING Plant_2_Generation_Data__ACI.csv: 1 valid rows disagree with 